# 04 — Gradient Episodic Memory (A-GEM)

GEM (Lopez-Paz & Ranzato, NeurIPS 2017) prevents forgetting by storing a small episodic memory
of past-task examples and constraining the current gradient not to increase the loss on those examples.

We implement the **A-GEM** variant (Chaudhry et al., ICLR 2019), which uses a single reference
gradient (the mean over all episodic memories) and projects the current gradient with a
closed-form formula — no QP solver required:

$$\tilde{g} = g - \frac{g \cdot g_{\text{ref}}}{\|g_{\text{ref}}\|^2} g_{\text{ref}} \quad \text{if } g \cdot g_{\text{ref}} < 0$$

This ensures gradient updates do not increase loss on past tasks stored in episodic memory.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.train import run_continual
from src.evaluate import backward_transfer, forward_transfer, average_accuracy, print_metrics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 1. Run A-GEM

In [ ]:
R_gem = run_continual(
    method='gem',
    dataset='cifar100',
    n_tasks=20,
    epochs_per_task=10,
    lr=0.1,
    batch_size=128,
    device=device,
    gem_memory=200,
    seed=42,
    save_dir='../results/baseline',
    verbose=True,
)

## 2. Accuracy Matrix

In [ ]:
T = R_gem.shape[0]
fig, ax = plt.subplots(figsize=(11, 9))
display = np.where(np.isnan(R_gem), 0.0, R_gem * 100)
im = ax.imshow(display, vmin=0, vmax=100, cmap='Blues', aspect='auto')
plt.colorbar(im, ax=ax, label='Accuracy (%)')
ax.set_xticks(range(T)); ax.set_yticks(range(T))
ax.set_xticklabels([f'T{j}' for j in range(T)], fontsize=8)
ax.set_yticklabels([f'T{i}' for i in range(T)], fontsize=8)
ax.set_xlabel('Task evaluated'); ax.set_ylabel('After training task')
ax.set_title('A-GEM — Accuracy Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/baseline/acc_matrix_gem.png', bbox_inches='tight')
plt.show()

## 3. Memory Size Ablation

In [ ]:
memory_sizes = [50, 100, 200, 500, 1000]
avg_accs, bwts = [], []

for mem in memory_sizes:
    print(f'  memory_per_task={mem} ...', end='')
    R = run_continual(
        method='gem', dataset='cifar100', n_tasks=5,
        epochs_per_task=5, lr=0.1, batch_size=128,
        device=device, gem_memory=mem, seed=42,
        save_dir='../results/baseline', verbose=False,
    )
    avg_accs.append(average_accuracy(R) * 100)
    bwts.append(backward_transfer(R) * 100)
    print(f' avg_acc={avg_accs[-1]:.1f}%  bwt={bwts[-1]:.1f}%')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(memory_sizes, avg_accs, marker='o', color='#3498db', linewidth=2)
ax1.set_xlabel('Memory per task'); ax1.set_ylabel('Avg Accuracy (%)')
ax1.set_title('A-GEM: Accuracy vs Memory Size'); ax1.grid(True, alpha=0.3)

ax2.plot(memory_sizes, bwts, marker='s', color='#e74c3c', linewidth=2)
ax2.set_xlabel('Memory per task'); ax2.set_ylabel('BWT (%)')
ax2.set_title('A-GEM: BWT vs Memory Size'); ax2.grid(True, alpha=0.3)
ax2.axhline(0, color='black', linewidth=0.8, linestyle='--')

plt.suptitle('A-GEM Memory Size Ablation (5 tasks)', fontweight='bold')
plt.tight_layout()
plt.savefig('../results/baseline/gem_memory_ablation.png', bbox_inches='tight')
plt.show()

## 4. Metrics

In [ ]:
print_metrics(R_gem, method='A-GEM (memory=200/task)')

## Observations

A-GEM provides better backward transfer than fine-tuning and is competitive with EWC on shorter
task sequences.  However it requires storing and replaying past data, which raises privacy concerns
in settings where past data cannot be retained (e.g. medical or personal data).
Pruning-based methods do not store any raw data.